# 004: Forward Propagation Mathematics — Vectorized matrix chain computations across layers

## 1. MATHEMATICAL DERIVATION
- For a network with $L$ layers, forward propagation computes:
  $$\mathbf{Z}^{[l]} = \mathbf{W}^{[l]} \mathbf{A}^{[l-1]} + \mathbf{b}^{[l]}, \quad l = 1, \ldots, L$$
  $$\mathbf{A}^{[l]} = g^{[l]}(\mathbf{Z}^{[l]})$$
- **Dimensions**: If layer $l$ has $n^{[l]}$ neurons and receives input from $n^{[l-1]}$ neurons across $m$ samples:
  - $\mathbf{W}^{[l]} \in \mathbb{R}^{n^{[l]} \times n^{[l-1]}}$
  - $\mathbf{b}^{[l]} \in \mathbb{R}^{n^{[l]} \times 1}$ (broadcast across $m$)
  - $\mathbf{Z}^{[l]}, \mathbf{A}^{[l]} \in \mathbb{R}^{n^{[l]} \times m}$

## 2. COMPUTATION GRAPH VIEW
- The forward pass forms a **Directed Acyclic Graph (DAG)** where each node is a tensor operation.
- This graph is later traversed in reverse during backpropagation.

---


In [ ]:
import numpy as np

def relu(Z):
    return np.maximum(0, Z)

def sigmoid(Z):
    return 1.0 / (1.0 + np.exp(-Z))

class ForwardPropNetwork:
    """Demonstrates vectorized forward propagation with dimension tracking."""
    def __init__(self, layer_dims):
        """layer_dims: list like [input, hidden1, hidden2, ..., output]."""
        np.random.seed(1)
        self.params = {}
        self.L = len(layer_dims) - 1  # Number of layers (excluding input)
        
        for l in range(1, self.L + 1):
            # He initialization: scale by sqrt(2 / fan_in)
            self.params[f'W{l}'] = np.random.randn(layer_dims[l], layer_dims[l-1]) * np.sqrt(2.0 / layer_dims[l-1])
            self.params[f'b{l}'] = np.zeros((layer_dims[l], 1))

    def forward(self, X):
        """Full vectorized forward pass. Returns final activation and cache."""
        caches = {}
        A = X  # A^[0] = input
        caches['A0'] = A
        
        for l in range(1, self.L + 1):
            W = self.params[f'W{l}']
            b = self.params[f'b{l}']
            
            # Linear step
            Z = W @ A + b  # Matrix multiplication + bias broadcast
            
            # Activation step
            if l == self.L:
                A = sigmoid(Z)  # Output layer: sigmoid for binary classification
            else:
                A = relu(Z)     # Hidden layers: ReLU
            
            caches[f'Z{l}'] = Z
            caches[f'A{l}'] = A
        
        return A, caches



In [ ]:
# ── Dimension verification across a 4-layer network ──
print("--- Forward Propagation Dimension Trace ---")
# Architecture: 5 inputs → 4 hidden → 3 hidden → 1 output
net = ForwardPropNetwork([5, 4, 3, 1])
X = np.random.randn(5, 10)  # 5 features, 10 samples

output, caches = net.forward(X)

for l in range(1, net.L + 1):
    W = net.params[f'W{l}']
    Z = caches[f'Z{l}']
    A = caches[f'A{l}']
    print(f"Layer {l}: W{l} {W.shape} @ A{l-1} {caches[f'A{l-1}'].shape} → Z{l} {Z.shape} → A{l} {A.shape}")

print(f"\nFinal output shape: {output.shape}")
print(f"Sample predictions: {output[0, :5].round(4)}")
